# Notebook 2: Standardized Prompt Preparation

## Research Project

**Evaluating the Functional Correctness and Consistency of AI-Generated Introductory Programming Solutions: Implications for Computing Education**

**Author:** Dr. C. V. Krishnaveni  
Department of Computer Science  
SKR & SKR Government College for Women (Autonomous), Kadapa

**GitHub Repository**
https://github.com/vkchennuru/AI-Code_Judgement

Notebook **2 of 7**

---

## Objective

Generate one standardized prompt for every benchmark programming problem.

The generated prompts will be used as inputs to the LLM in Notebook 3.

### Input

- dataset/cs1_problems_clean.csv

### Output

- prompts/problem_001_prompt.txt
- prompts/problem_002_prompt.txt
- …
- prompts/prompt_manifest.csv

---

## Workflow

1. Clone GitHub repository
2. Load benchmark dataset
3. Validate dataset
4. Create prompts directory
5. Generate prompts
6. Verify prompts
7. Create prompt manifest
8. Save outputs

## Requirements

Python Packages

- pandas
- pathlib

Google Colab already contains these packages.

If running locally:

```bash
pip install pandas
```

In [1]:
# ============================================================
# Clone GitHub Repository
# ============================================================

!rm -rf AI-Code_Judgement

!git clone https://github.com/vkchennuru/AI-Code_Judgement.git

Cloning into 'AI-Code_Judgement'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 149 (delta 53), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 377.26 KiB | 5.89 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [2]:
from pathlib import Path
import pandas as pd

In [3]:
PROJECT_ROOT = Path("/content/AI-Code_Judgement")

DATASET_DIR = PROJECT_ROOT / "dataset"

PROMPTS_DIR = PROJECT_ROOT / "prompts"

PROMPTS_DIR.mkdir(exist_ok=True)

print(PROJECT_ROOT)
print(DATASET_DIR)
print(PROMPTS_DIR)

/content/AI-Code_Judgement
/content/AI-Code_Judgement/dataset
/content/AI-Code_Judgement/prompts


## Step 1: Load the Clean Benchmark Dataset

The benchmark dataset produced by Notebook 1 is loaded and validated before prompt generation.

In [7]:
df = pd.read_csv(DATASET_DIR / "cs1_problems.csv")

print("✓ Dataset Loaded Successfully")

print(f"Number of Problems : {len(df)}")

print(f"Number of Columns : {len(df.columns)}")

display(df.head())

✓ Dataset Loaded Successfully
Number of Problems : 10
Number of Columns : 12


,problem_id,title,topic,difficulty,problem_statement,input_format,output_format,constraints,sample_input,sample_output,expected_algorithm,concepts
0,1,Sum of Two Integers,Variables,Easy,Write a Python program that reads two integers...,Two integers separated by a space.,Print a single integer representing their sum.,"-10^9 ? a, b ? 10^9",5 10,15,Addition,"Variables, Input, Output"
1,2,Area of a Circle,Operators,Easy,Write a Python program that reads the radius o...,One positive real number representing the radius.,Print the area rounded to two decimal places.,0 ? radius ? 10^4,5,78.54,Formula (?r²),"Operators, Arithmetic"
2,3,Largest of Two Numbers,Selection,Easy,Write a Python program that reads two integers...,Two integers separated by a space.,Print the larger integer.,"-10^9 ? a, b ? 10^9",12 7,12,Conditional Selection,"If Statement, Comparison"
3,4,Print Numbers from 1 to N,Loops,Easy,Write a Python program that reads an integer N...,One positive integer N.,Numbers from 1 to N separated by spaces.,1 ? N ? 1000,5,1 2 3 4 5,Iteration,Loops
4,5,Factorial of a Number,Loops,Easy,Write a Python program that reads a non-negati...,One non-negative integer.,Print the factorial value.,0 ? N ? 20,5,120,Iterative Multiplication,Loops


## Step 2: Validate the Benchmark Dataset

This section verifies that the benchmark dataset has been loaded correctly before generating prompts.

In [8]:
print("=" * 60)
print("Benchmark Dataset Validation")
print("=" * 60)

print(f"Total Problems : {len(df)}")
print(f"Total Columns  : {len(df.columns)}")

print("\nColumns:")

for col in df.columns:
    print("✓", col)

Benchmark Dataset Validation
Total Problems : 10
Total Columns  : 12

Columns:
✓ problem_id
✓ title
✓ topic
✓ difficulty
✓ problem_statement
✓ input_format
✓ output_format
✓ constraints
✓ sample_input
✓ sample_output
✓ expected_algorithm
✓ concepts


In [9]:
display(df.head())

,problem_id,title,topic,difficulty,problem_statement,input_format,output_format,constraints,sample_input,sample_output,expected_algorithm,concepts
0,1,Sum of Two Integers,Variables,Easy,Write a Python program that reads two integers...,Two integers separated by a space.,Print a single integer representing their sum.,"-10^9 ? a, b ? 10^9",5 10,15,Addition,"Variables, Input, Output"
1,2,Area of a Circle,Operators,Easy,Write a Python program that reads the radius o...,One positive real number representing the radius.,Print the area rounded to two decimal places.,0 ? radius ? 10^4,5,78.54,Formula (?r²),"Operators, Arithmetic"
2,3,Largest of Two Numbers,Selection,Easy,Write a Python program that reads two integers...,Two integers separated by a space.,Print the larger integer.,"-10^9 ? a, b ? 10^9",12 7,12,Conditional Selection,"If Statement, Comparison"
3,4,Print Numbers from 1 to N,Loops,Easy,Write a Python program that reads an integer N...,One positive integer N.,Numbers from 1 to N separated by spaces.,1 ? N ? 1000,5,1 2 3 4 5,Iteration,Loops
4,5,Factorial of a Number,Loops,Easy,Write a Python program that reads a non-negati...,One non-negative integer.,Print the factorial value.,0 ? N ? 20,5,120,Iterative Multiplication,Loops


In [10]:
print(df.columns.tolist())

['problem_id', 'title', 'topic', 'difficulty', 'problem_statement', 'input_format', 'output_format', 'constraints', 'sample_input', 'sample_output', 'expected_algorithm', 'concepts']


## Step 3: Create the Prompt Directory

This section creates the output directory where all generated prompt files will be stored.

In [12]:
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Prompt directory is ready.")
print(PROMPTS_DIR)

✓ Prompt directory is ready.
/content/AI-Code_Judgement/prompts


## Step 4: Define the Standard Prompt Template

A standardized prompt template is used to ensure that every benchmark problem is presented to the LLM in an identical format.

In [13]:
def create_prompt(row):
    prompt = f"""
You are an expert Python programmer.

Solve the following programming problem.

Problem ID:
{row['problem_id']}

Title:
{row['title']}

Topic:
{row['topic']}

Difficulty:
{row['difficulty']}

Problem Statement:
{row['problem_statement']}

Input Format:
{row['input_format']}

Output Format:
{row['output_format']}

Constraints:
{row['constraints']}

Sample Input:
{row['sample_input']}

Sample Output:
{row['sample_output']}

Expected Algorithm:
{row['expected_algorithm']}

Concepts:
{row['concepts']}

Instructions

1. Write only Python 3 code.
2. Do not include explanations.
3. Do not include markdown.
4. Produce executable code only.
"""

    return prompt.strip()

## Step 5: Generate Prompt Files

Each benchmark problem is converted into a standardized prompt and saved as an individual text file.

In [14]:
# ============================================================
# Generate Prompt Files
# ============================================================

prompt_records = []

for _, row in df.iterrows():

    prompt = create_prompt(row)

    filename = f"problem_{int(row['problem_id']):03d}_prompt.txt"

    filepath = PROMPTS_DIR / filename

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(prompt)

    prompt_records.append({
        "problem_id": row["problem_id"],
        "filename": filename
    })

print(f"✓ Generated {len(prompt_records)} prompt files.")

✓ Generated 10 prompt files.


## Step 6: Verify Generated Prompt Files

In [15]:
prompt_files = sorted(PROMPTS_DIR.glob("*.txt"))

print("=" * 60)
print("Prompt Generation Summary")
print("=" * 60)

print(f"Prompt files generated : {len(prompt_files)}")

Prompt Generation Summary
Prompt files generated : 10


In [16]:
assert len(prompt_files) == len(df)

print("✓ Verification Successful")
print("Every programming problem has one prompt file.")

✓ Verification Successful
Every programming problem has one prompt file.


## Step 7: Preview a Generated Prompt

The following cell displays one generated prompt to verify its structure before it is used for code generation.

In [17]:
sample_prompt = prompt_files[0]

print(f"Previewing: {sample_prompt.name}\n")
print("=" * 80)

with open(sample_prompt, "r", encoding="utf-8") as f:
    print(f.read())

Previewing: problem_001_prompt.txt

You are an expert Python programmer.

Solve the following programming problem.

Problem ID:
1

Title:
Sum of Two Integers

Topic:
Variables

Difficulty:
Easy

Problem Statement:
Write a Python program that reads two integers separated by a space and prints their sum.

Input Format:
Two integers separated by a space.

Output Format:
Print a single integer representing their sum.

Constraints:
-10^9 ? a, b ? 10^9

Sample Input:
5 10

Sample Output:
15

Expected Algorithm:
Addition

Concepts:
Variables, Input, Output

Instructions

1. Write only Python 3 code.
2. Do not include explanations.
3. Do not include markdown.
4. Produce executable code only.


## Step 8: Create the Prompt Manifest

A manifest file is created to map each programming problem to its corresponding prompt file.

In [18]:
manifest_df = pd.DataFrame(prompt_records)

manifest_path = PROMPTS_DIR / "prompt_manifest.csv"

manifest_df.to_csv(manifest_path, index=False)

print("✓ Prompt manifest saved successfully.")
print(manifest_path)

✓ Prompt manifest saved successfully.
/content/AI-Code_Judgement/prompts/prompt_manifest.csv


In [19]:
display(manifest_df)

,problem_id,filename
0,1,problem_001_prompt.txt
1,2,problem_002_prompt.txt
2,3,problem_003_prompt.txt
3,4,problem_004_prompt.txt
4,5,problem_005_prompt.txt
5,6,problem_006_prompt.txt
6,7,problem_007_prompt.txt
7,8,problem_008_prompt.txt
8,9,problem_009_prompt.txt
9,10,problem_010_prompt.txt


In [20]:
print("=" * 60)
print("Prompt Manifest Summary")
print("=" * 60)

print(f"Total prompt records : {len(manifest_df)}")
print(f"Manifest file        : {manifest_path.name}")

Prompt Manifest Summary
Total prompt records : 10
Manifest file        : prompt_manifest.csv


In [21]:
assert len(manifest_df) == len(df)

print("✓ Final Verification Successful")
print("All benchmark problems have corresponding prompt files and manifest entries.")

✓ Final Verification Successful
All benchmark problems have corresponding prompt files and manifest entries.


## Reproducibility

This notebook generates standardized prompt files from the benchmark dataset. Running the notebook multiple times with the same dataset produces identical prompt files, ensuring reproducibility of the experimental workflow.

## Conclusion

In this notebook:

- The benchmark dataset was loaded successfully.
- Standardized prompts were generated for each programming problem.
- Individual prompt files were saved in the `prompts/` directory.
- A prompt manifest was created to map benchmark problems to prompt files.
- The prompt generation process was verified successfully.

These prompts serve as the standardized inputs for the AI code generation stage in Notebook 3.

In [22]:
print("=" * 70)
print("Notebook 2 Completed Successfully")
print("=" * 70)
print("Output Directory :", PROMPTS_DIR)
print("Prompt Files     :", len(prompt_files))
print("Manifest File    :", manifest_path.name)
print("Ready for Notebook 3: AI Code Generation")

Notebook 2 Completed Successfully
Output Directory : /content/AI-Code_Judgement/prompts
Prompt Files     : 10
Manifest File    : prompt_manifest.csv
Ready for Notebook 3: AI Code Generation
